In [5]:
import os
import sys
import pandas as pd
import geopandas as gpd
import libpysal
import geoplanar
import momepy
from importlib import reload
from tqdm import tqdm
import numpy as np

In [6]:
sys.path.append("../src")

In [7]:
import main

# Instructions

When running the notebook for the first time

0. [Calculate 2-dimensional parameters](#calculate-2-dimensional-parameters)

Otherwise,
1. [Import 2d parameters](#import-2d-parameters)
2. [Import 3d parameters](#import-3d-parameters)
3. [Aggregate to station buffers](#aggregate-to-station-buffers)

# Calculate 2-dimensional parameters

In [8]:
streets=gpd.read_parquet('../data/processed_data/preprocessed_streets.parquet')
buildings = gpd.read_parquet('../data/processed_data/preprocessed_buildings_v2.parquet')

In [ ]:
buildings

In [ ]:
m = streets.explore()
nodes.explore(m=m)

In [10]:
def merge_touching(gdf):
    source, target = gdf.boundary.sindex.query(
        gdf.boundary, predicate="overlaps"
    )

    neighbors = {}
    for i in gdf.index:
        if i in source:
            neighbors[i] = list(target[source == i])
        else:
            neighbors[i] = []  # isolated polygon → singleton component

    w = libpysal.graph.Graph.from_dicts(neighbors)

    dissolved_gdf = gdf.dissolve(by=w.component_labels)

    dissolved_gdf.index = (
        w.component_labels
        .drop_duplicates()
        .index
    )

    dissolved_gdf = dissolved_gdf.rename_axis(index=gdf.index.name)
    return dissolved_gdf


In [11]:
dissolved_buildings = merge_touching(buildings)

In [ ]:
dissolved_buildings

In [ ]:
dissolved_buildings.explore()

In [ ]:
dissolved_buildings.columns

In [9]:
dissolved_buildings = dissolved_buildings.reset_index()

In [ ]:
dissolved_buildings.area.sum()

In [ ]:
reload(main)

In [ ]:
output = main.block_params(dissolved_buildings,dissolved_buildings['measuredHeight'],streets)

In [13]:
bldgs, streets, nodes = output

In [14]:
bldgs.to_parquet('../data/processed_data/processed_buildings_merged_blocks.parquet')
streets = streets.set_geometry('geometry_x')
streets.to_parquet('../data/processed_data/processed_streets_merged_blocks.parquet')
nodes.to_parquet('../data/processed_data/processed_nodes_merged_blocks.parquet')

# Import 2D Parameters

In [13]:
bldgs = gpd.read_parquet('../data/processed_data/processed_buildings_merged_blocks.parquet')
streets = gpd.read_parquet('../data/processed_data/processed_streets_merged_blocks.parquet')
nodes = gpd.read_parquet('../data/processed_data/processed_nodes_merged_blocks.parquet')

In [ ]:
bldgs.BuAre.median()

# Import 3D parameters

In [15]:
stats_3D = pd.read_csv('../data/raw_data/LoD1_stats.csv')

In [ ]:
stats_3D[['point_count', 'unique_point_count',
       'surface_count', 'actual_volume', 'convex_hull_volume', 'obb_volume',
       'aabb_volume', 'footprint_perimeter', 'obb_width', 'obb_length',
       'surface_area', 'ground_area', 'wall_area', 'roof_area',
       'ground_point_count', 'wall_point_count', 'roof_point_count',
       'ground_surface-count', 'wall_surface_count', 'roof_surface_count',]]

In [ ]:
stats_3D.columns

In [18]:
remap = {'surface_count':'BuSurf_3D', 'actual_volume':'BuVol_3D',
       'surface_area':'BuSA_3D', 'circularity_2d':'BuCir', 'hemisphericality_3d':'BuHem_3D',
       'convexity_3d':'BuCon_3D', 'fractality_2d':'BuFra', 'fractality_3d':'BuFra_3D',
       'rectangularity_3d':'BuCubo_3D', 'squareness_2d':'BuSqu',
       'cubeness_3d':'BuCube_3D', 'min_vertical_elongation':'BumVE_3D',
       'max_vertical_elongation':'BuMVE_3D', 'form_factor_3D':'BuFF_3D',
       'equivalent_prism_index_3d':'BuEPI_3D',
       'proximity_index_2d_':'BuProx', 'proximity_index_3d':'BuProx_3D', 'exchange_index_2d':'BuEx',
       'exchange_index_3d':'BuEx_3D', 'spin_index_2d':'BuSpi', 'spin_index_3d':'BuSpi_3D',
       'perimeter_index_2d':'BuPerC', 'circumference_index_3d':'BuCf_3D', 'depth_index_2d':'BuDep',
       'depth_index_3d':'BuDep_3D', 'girth_index_2d':'BuGir', 'girth_index_3d':'BuGir_3D',
       'dispersion_index_2d':'BuDisp', 'dispersion_index_3d':'BuDisp_3D', 'range_index_2d':'BuRan',
       'range_index_3d':'BuRan_3D', 'roughness_index_2d':'BuRough', 'roughness_index_3d':'BuRough_3D',
       'shared_walls_area':'BuSWA_3D'}

In [19]:
stats_3D = stats_3D.rename(columns=remap)

In [20]:
stats_3D['BuSWR_3D'] = stats_3D['BuSWA_3D'] / stats_3D['BuSA_3D']

In [21]:
all_buildings = bldgs.merge(stats_3D,left_on='oid',right_on='id',how='left', suffixes=('_2D', '_3D'))

In [22]:
# Exposed Wall Area
all_buildings['BuEWA_3D'] = all_buildings['BuSA_3D'] - all_buildings['BuSWA_3D'] - all_buildings['BuAre']
all_buildings['BuEWR_3D'] = all_buildings['BuEWA_3D'] / all_buildings['BuSA_3D']

In [23]:
all_buildings.to_parquet('../data/processed_data/processed_buildings_merged_blocks_3D.parquet')

## Aggregate to station buffers

In [24]:
bldgs = gpd.read_parquet('../data/processed_data/processed_buildings_merged_blocks_3D.parquet')

In [ ]:
bldgs.explore()

In [ ]:
bldgs['measuredHeight'].max()

In [ ]:
# list tallest buildings freiburg
bldgs.sort_values('measuredHeight', ascending=False).head(10)['measuredHeight']

In [ ]:
bldgs.sort_values('measuredHeight', ascending=False).head(10)

### Select stations where elevation is no more than 100m above elevation of the central station

In [19]:
stations = pd.read_csv("../data/raw_data/Freiburg-Street-Level-Weather-Station-Network-MetaData-V1-0.csv")
stations['station_elevation_diff'] = stations['station_elevation'] - stations[stations['station_id'] == 'HBHF']['station_elevation'].values[0]
stations = stations[stations['station_elevation_diff'] < 100]

In [ ]:
stations

### If SVF needs to be masked

In [ ]:
svf_path = '../data/processed_data/SVF.tif'
# mask with building footprints for street-level SVF
masked_svf_path = '../data/processed_data/SVF_masked.tif'
main.mask_raster(svf_path, bldgs, masked_svf_path)

In [ ]:
# Aggregate parameters for all stations for all radii below

### Aggregate parameters for all stations for all radii below

In [28]:
radii = [30,40,50,60,70,80,90,100,110,120,130,140,150,160,170,180,190,200,210,220,230,240,250,260,270,280,290,300,320,340,360,380,400,450,500,600,750,1000,1250,1500,1750,2000]

In [ ]:
streets

In [ ]:
reload(main)

In [ ]:
bldgs.explore()

In [17]:
radii = [300]

In [ ]:
for i in tqdm(radii):
    print('Processing radius: ' + str(i))
    stn_buffers = main.buffer_stations(stations, radius=i)
    stn_buffers = main.neighbourhood_graph_params(bldgs, stn_buffers)
    b,s,n = main.select_objects(bldgs, streets, nodes, stn_buffers)
    stn_buffers = main.aggregate_params(b,s,n, stn_buffers)
    
    masked_svf_path = '../data/processed_data/SVF_street_buildings_only.tif'
    stn_buffers = main.agg_raster(masked_svf_path, stn_buffers, 'SVF_3D')

    stn_buffers['HW_v2'] = stn_buffers['BuHt_wmean'] / stn_buffers['BuIBD']
    ref_area = np.pi * i**2
    stn_buffers['PAI'] = stn_buffers['BuAre_sum'] / ref_area
    stn_buffers['HW_del'] = stn_buffers['BuHt_wmean'] / stn_buffers['BuW_delaunay_mean']
    stn_buffers['HW_n1'] = stn_buffers['BuHt_wmean'] / stn_buffers['BuW_knn1_mean']
    stn_buffers['HW_n5'] = stn_buffers['BuHt_wmean'] / stn_buffers['BuW_knn5_mean']

    stn_buffers.to_parquet('../data/processed_data/processed_station_params_merged_blocks_' + str(i) + '.parquet')

In [ ]:
stn_buffers['BuIBD']